# GDS Lab: kNN Aircraft Similarity from Sensor + Maintenance Features

> **Optional GDS Example.** This notebook is an optional example demonstrating the Neo4j Graph Data Science (GDS) library. It is not required to complete the workshop. Run it if you want to explore graph algorithms on the aircraft dataset.

> ⚠️ **Requires AuraDB Professional.** The Neo4j Graph Data Science (GDS) library used in this notebook is only available on AuraDB Professional (or higher). It is **not** included in the AuraDB Free tier, so this notebook will not run on a free instance.

This notebook demonstrates a **dual-database enrichment loop**: compute per-aircraft
feature vectors from Databricks sensor telemetry, write those features back to Neo4j
Aircraft nodes, then run GDS k-Nearest Neighbors to find peer aircraft with similar
operational profiles.

## What You'll Learn
- How to compute multi-feature vectors from Delta Lake sensor data
- How to write computed features back to Neo4j node properties via Spark Connector
- How to project a node-property graph and run kNN in GDS
- How to write `SIMILAR_PROFILE` relationships and query them as a peer-alert pattern

## Core Idea
Each aircraft gets a feature vector of 7 normalized metrics spanning sensor behavior
and maintenance burden. kNN finds the 3 most similar peers for every aircraft.
The operational use case: when aircraft X triggers a fault alert, proactively
inspect its kNN peers — they share the same degradation signature.

## Prerequisites
- Neo4j Aura credentials (AuraDB Professional or higher — GDS plugin required)
- Full dataset loaded via `01_aircraft_etl_to_neo4j.ipynb`
- Sensor readings Delta tables available (from workshop setup)

## Instructions
1. Enter your credentials in Section 1
2. Set CATALOG and SCHEMA to match your workshop environment
3. Run all cells in order

## Section 1: Configuration

In [ ]:
%pip install neo4j

In [ ]:
# ============================================
# CONFIGURATION - Enter your credentials
# ============================================

NEO4J_URI      = ""      # e.g., "neo4j+s://xxxxxxxx.databases.neo4j.io"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = ""      # Your password from Lab 1
NEO4J_DATABASE = "neo4j"  # Neo4j database to use (Aura default is "neo4j")

# Databricks Unity Catalog location of sensor Delta tables
CATALOG  = "databricks-neo4j-workshop"
SCHEMA   = "aircraft"

# Unity Catalog Volume path for CSV files
DATA_PATH = "/Volumes/databricks-neo4j-workshop/aircraft/raw_data"

if not NEO4J_URI or not NEO4J_PASSWORD:
    print("WARNING: Please enter your Neo4j credentials above before running!")
else:
    print(f"Neo4j: {NEO4J_URI}")
    print(f"Delta: {CATALOG}.{SCHEMA}")

In [ ]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

def run_query(cypher, params=None):
    """Execute a Cypher query and return results as a list of dicts."""
    with driver.session(database=NEO4J_DATABASE) as session:
        return [dict(r) for r in session.run(cypher, params or {})]

# Configure Spark Connector
spark.conf.set("neo4j.url", NEO4J_URI)
spark.conf.set("neo4j.authentication.basic.username", NEO4J_USERNAME)
spark.conf.set("neo4j.authentication.basic.password", NEO4J_PASSWORD)
spark.conf.set("neo4j.database", NEO4J_DATABASE)

def spark_query(cypher):
    """Read Neo4j query results as a Spark DataFrame."""
    return (spark.read
        .format("org.neo4j.spark.DataSource")
        .option("query", cypher)
        .load())

result = run_query("RETURN 'Connected to Neo4j' AS status")
print(result[0]["status"])

## Section 2: Compute Feature Vectors in Databricks

We compute 7 features per aircraft:

| Feature | Source | Signal |
|---|---|---|
| `avg_egt` | EGT sensor mean over 90 days | Baseline engine temperature |
| `stddev_egt` | EGT sensor std dev | Temperature variability / instability |
| `avg_vibration` | Vibration sensor mean | Baseline mechanical stress |
| `stddev_vibration` | Vibration sensor std dev | Vibration variability — key anomaly indicator |
| `avg_fuel_flow` | FuelFlow sensor mean | Fuel efficiency proxy |
| `total_events` | Maintenance event count | Overall maintenance burden |
| `critical_events` | Critical-severity events | High-risk event concentration |

Std dev features are deliberately included: two aircraft can have the same average EGT
but very different stability — the unstable one is more likely failing.

In [ ]:
# Sensor aggregates from Delta tables
sensor_stats = spark.sql(f"""
    SELECT
        s.aircraft_id,
        AVG  (CASE WHEN sen.type = 'EGT'       THEN r.value END) AS avg_egt,
        STDDEV(CASE WHEN sen.type = 'EGT'       THEN r.value END) AS stddev_egt,
        AVG  (CASE WHEN sen.type = 'Vibration'  THEN r.value END) AS avg_vibration,
        STDDEV(CASE WHEN sen.type = 'Vibration'  THEN r.value END) AS stddev_vibration,
        AVG  (CASE WHEN sen.type = 'FuelFlow'   THEN r.value END) AS avg_fuel_flow
    FROM `{CATALOG}`.`{SCHEMA}`.sensor_readings r
    JOIN `{CATALOG}`.`{SCHEMA}`.sensors  sen ON r.sensor_id  = sen.sensor_id
    JOIN `{CATALOG}`.`{SCHEMA}`.systems  s   ON sen.system_id = s.system_id
    GROUP BY s.aircraft_id
""")

print(f"Sensor features computed for {sensor_stats.count()} aircraft")
display(sensor_stats)

In [ ]:
from pyspark.sql.functions import col, count, when

# Maintenance event counts from CSV (maintenance data lives only in Neo4j/Volume)
maintenance_df = (spark.read
    .option("header", "true")
    .csv(f"{DATA_PATH}/nodes_maintenance.csv")
    .withColumnRenamed(":ID(MaintenanceEvent)", "event_id"))

maintenance_stats = (maintenance_df
    .groupBy("aircraft_id")
    .agg(
        count("*").alias("total_events"),
        count(when(col("severity") == "CRITICAL", True)).alias("critical_events")
    ))

print(f"Maintenance features computed for {maintenance_stats.count()} aircraft")
display(maintenance_stats)

In [ ]:
# Join sensor and maintenance features
features_df = sensor_stats.join(maintenance_stats, "aircraft_id", "left").fillna(0)

print(f"Combined feature matrix: {features_df.count()} aircraft, {len(features_df.columns)} columns")
display(features_df)

In [ ]:
from pyspark.sql.functions import min as spark_min, max as spark_max, lit

FEATURE_COLS = [
    "avg_egt", "stddev_egt",
    "avg_vibration", "stddev_vibration",
    "avg_fuel_flow",
    "total_events", "critical_events"
]

def minmax_normalize(df, columns):
    """Min-max normalize each column into a *_norm variant (range 0-1)."""
    result = df
    for c in columns:
        stats = df.agg(spark_min(c).alias("lo"), spark_max(c).alias("hi")).collect()[0]
        lo, hi = stats["lo"], stats["hi"]
        if hi > lo:
            result = result.withColumn(f"{c}_norm", (col(c) - lo) / (hi - lo))
        else:
            result = result.withColumn(f"{c}_norm", lit(0.0))
    return result

features_norm = minmax_normalize(features_df, FEATURE_COLS)

norm_cols = ["aircraft_id"] + [f"{c}_norm" for c in FEATURE_COLS]
display(features_norm.select(norm_cols))

## Section 3: Write Features to Neo4j Aircraft Nodes

The Spark Connector writes the normalized feature columns as properties on each
Aircraft node, matched by `aircraft_id`. GDS will read these properties during
graph projection in Section 4.

In [ ]:
NORM_COLS = [f"{c}_norm" for c in FEATURE_COLS]

(features_norm
    .select(["aircraft_id"] + NORM_COLS)
    .write
    .format("org.neo4j.spark.DataSource")
    .mode("Overwrite")
    .option("labels", ":Aircraft")
    .option("node.keys", "aircraft_id")
    .save())

print(f"Written {len(NORM_COLS)} feature properties to Aircraft nodes.")

In [ ]:
# Verify features landed on Aircraft nodes
sample = run_query("""
    MATCH (a:Aircraft)
    WHERE a.avg_egt_norm IS NOT NULL
    RETURN a.tail_number      AS tail_number,
           round(a.avg_egt_norm, 3)       AS avg_egt_norm,
           round(a.stddev_vibration_norm, 3) AS stddev_vib_norm,
           round(a.critical_events_norm, 3) AS critical_events_norm
    ORDER BY a.tail_number
    LIMIT 8
""")

print(f"{'Aircraft':<12} {'avg_egt_norm':>14} {'stddev_vib_norm':>16} {'critical_norm':>14}")
print("-" * 58)
for r in sample:
    print(f"{r['tail_number']:<12} {r['avg_egt_norm']:>14} {r['stddev_vib_norm']:>16} {r['critical_events_norm']:>14}")

## Section 4: Project the Aircraft Graph and Run kNN

**k-Nearest Neighbors (kNN)** computes pairwise similarity between nodes using
their property vectors. We project only Aircraft nodes with no relationships —
kNN operates purely on node properties.

Similarity is computed as cosine similarity across all 7 normalized features.
`topK: 3` means each aircraft gets its 3 most similar peers.

In [ ]:
result = run_query("RETURN gds.version() AS version")
print(f"GDS version: {result[0]['version']}")

In [ ]:
run_query("CALL gds.graph.drop('aircraft-profiles', false) YIELD graphName")
print("Cleared previous projection.")

In [ ]:
# Project Aircraft nodes with their feature properties.
# kNN scores on node-property vectors only, but GDS requires at least one
# relationship type in a projection, so we pass '*' (any Aircraft-to-Aircraft
# edges are ignored by the algorithm).
result = run_query("""
    CALL gds.graph.project(
        'aircraft-profiles',
        {
            Aircraft: {
                properties: [
                    'avg_egt_norm', 'stddev_egt_norm',
                    'avg_vibration_norm', 'stddev_vibration_norm',
                    'avg_fuel_flow_norm',
                    'total_events_norm', 'critical_events_norm'
                ]
            }
        },
        '*'
    )
    YIELD graphName, nodeCount, relationshipCount
""")

proj = result[0]
print(f"Projection '{proj['graphName']}': {proj['nodeCount']} nodes, {proj['relationshipCount']} relationships")

In [ ]:
# Stream kNN results — inspect peer assignments before writing.
# concurrency: 1 is required whenever randomSeed is set, so results are reproducible.
knn_results = run_query("""
    CALL gds.knn.stream('aircraft-profiles', {
        topK: 3,
        nodeProperties: [
            'avg_egt_norm', 'stddev_egt_norm',
            'avg_vibration_norm', 'stddev_vibration_norm',
            'avg_fuel_flow_norm',
            'total_events_norm', 'critical_events_norm'
        ],
        similarityCutoff: 0.4,
        randomSeed: 42,
        concurrency: 1
    })
    YIELD node1, node2, similarity
    RETURN gds.util.asNode(node1).tail_number AS aircraft,
           gds.util.asNode(node1).model       AS model,
           gds.util.asNode(node2).tail_number AS peer_aircraft,
           gds.util.asNode(node2).model       AS peer_model,
           round(similarity, 4)               AS similarity_score
    ORDER BY aircraft, similarity_score DESC
""")

current = None
for r in knn_results:
    if r["aircraft"] != current:
        current = r["aircraft"]
        print(f"\n{r['aircraft']} ({r['model']}) — top peers:")
        print(f"  {'Peer':<12} {'Model':<12} {'Similarity':>12}")
        print(f"  {'-'*38}")
    print(f"  {r['peer_aircraft']:<12} {r['peer_model']:<12} {r['similarity_score']:>12}")

## Section 5: Write SIMILAR_PROFILE Relationships to Neo4j

Writing the similarity scores as relationships lets us query the peer network
directly in Cypher and visualize it in the Neo4j Browser.

In [ ]:
# concurrency: 1 is required whenever randomSeed is set, so the written
# relationships match the streamed preview above.
result = run_query("""
    CALL gds.knn.write('aircraft-profiles', {
        topK: 3,
        writeRelationshipType: 'SIMILAR_PROFILE',
        writeProperty: 'similarity_score',
        nodeProperties: [
            'avg_egt_norm', 'stddev_egt_norm',
            'avg_vibration_norm', 'stddev_vibration_norm',
            'avg_fuel_flow_norm',
            'total_events_norm', 'critical_events_norm'
        ],
        randomSeed: 42,
        concurrency: 1
    })
    YIELD relationshipsWritten, nodesCompared
""")

r = result[0]
print(f"Compared {r['nodesCompared']} aircraft pairs")
print(f"Wrote {r['relationshipsWritten']} SIMILAR_PROFILE relationships to Neo4j")

In [ ]:
# Verify the relationships exist
verify = run_query("""
    MATCH ()-[r:SIMILAR_PROFILE]-()
    RETURN count(r) AS relationship_count,
           round(avg(r.similarity_score), 4) AS avg_similarity,
           round(min(r.similarity_score), 4) AS min_similarity,
           round(max(r.similarity_score), 4) AS max_similarity
""")

v = verify[0]
print(f"SIMILAR_PROFILE relationships: {v['relationship_count']}")
print(f"Similarity range: {v['min_similarity']} – {v['max_similarity']}  (avg {v['avg_similarity']})")

## Section 6: Peer Alert Pattern

The operational use case: when aircraft X shows an anomalous sensor reading,
proactively inspect its kNN peers. They share the same feature signature and
may be on a similar degradation trajectory.

In [ ]:
# Which peers should we inspect when a given aircraft flags an anomaly?
TARGET_AIRCRAFT = "N10000"  # Change to any tail number from the fleet

peers = run_query("""
    MATCH (a:Aircraft {tail_number: $tail})-[r:SIMILAR_PROFILE]->(peer:Aircraft)
    RETURN peer.tail_number AS peer_tail,
           peer.model       AS peer_model,
           peer.operator    AS peer_operator,
           round(r.similarity_score, 4) AS similarity
    ORDER BY similarity DESC
""", {"tail": TARGET_AIRCRAFT})

print(f"When {TARGET_AIRCRAFT} flags an anomaly, inspect these peer aircraft:\n")
print(f"  {'Tail Number':<14} {'Model':<12} {'Operator':<14} {'Similarity':>12}")
print(f"  {'-'*55}")
for r in peers:
    print(f"  {r['peer_tail']:<14} {r['peer_model']:<12} {r['peer_operator']:<14} {r['similarity']:>12}")

In [ ]:
# Read the full similarity network into Databricks for downstream use
similarity_network = spark_query("""
    MATCH (a:Aircraft)-[r:SIMILAR_PROFILE]->(peer:Aircraft)
    RETURN a.aircraft_id    AS aircraft_id,
           a.tail_number    AS aircraft,
           a.model          AS model,
           peer.tail_number AS peer_aircraft,
           peer.model       AS peer_model,
           round(r.similarity_score, 4) AS similarity_score
    ORDER BY aircraft, similarity_score DESC
""")

display(similarity_network)

In [ ]:
# Compare feature vectors for a target aircraft and its top peer
# Confirm the similarity score is grounded in meaningful feature overlap
top_peer = similarity_network.filter(
    f"aircraft = '{TARGET_AIRCRAFT}'"
).orderBy("similarity_score", ascending=False).first()

if top_peer:
    # features_norm is keyed by aircraft_id (e.g., AC1001), so map the tail
    # numbers back to aircraft_ids via the similarity network
    tail_to_id = {
        row["aircraft"]: row["aircraft_id"]
        for row in similarity_network.select("aircraft", "aircraft_id").distinct().collect()
    }
    comparison_ids = [tail_to_id[TARGET_AIRCRAFT], tail_to_id[top_peer["peer_aircraft"]]]
    comparison = features_norm.filter(
        col("aircraft_id").isin(comparison_ids)
    ).select(["aircraft_id"] + NORM_COLS)
    display(comparison)
else:
    print(f"No peers found for {TARGET_AIRCRAFT} — try a different tail number.")

## Section 7: Clean Up

In [ ]:
# Drop the projection (SIMILAR_PROFILE relationships and feature properties persist)
run_query("CALL gds.graph.drop('aircraft-profiles', false) YIELD graphName")
driver.close()
print("Projection dropped.")
print("SIMILAR_PROFILE relationships and *_norm properties remain on Aircraft nodes.")

## Neo4j Browser Queries

**Visualize the similarity network:**
```cypher
MATCH (a:Aircraft)-[r:SIMILAR_PROFILE]->(peer:Aircraft)
RETURN a, r, peer
```

**Find the highest-similarity pairs:**
```cypher
MATCH (a:Aircraft)-[r:SIMILAR_PROFILE]->(peer:Aircraft)
RETURN a.tail_number AS aircraft, peer.tail_number AS peer,
       a.model AS model_a, peer.model AS model_peer,
       round(r.similarity_score, 4) AS similarity
ORDER BY similarity DESC
LIMIT 10
```

**Cross-model similarity — which aircraft cross model boundaries?**
```cypher
MATCH (a:Aircraft)-[r:SIMILAR_PROFILE]->(peer:Aircraft)
WHERE a.model <> peer.model
RETURN a.tail_number AS aircraft, a.model AS model,
       peer.tail_number AS peer, peer.model AS peer_model,
       round(r.similarity_score, 4) AS similarity
ORDER BY similarity DESC
```